# **Práctica 4**: GANs
# Parte 1: GANs simples

Simplificación de:
https://medium.com/@mattiaspinelli/simple-generative-adversarial-network-gans-with-keras-1fe578e44a87

### Ejercicio prelab:
Lee y ejecuta el siguiente código

Importamos librerías

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # Suppress info messages (already set in Colab)
import tensorflow as tf

In [ ]:
import numpy as np

from keras.datasets import mnist
from keras.layers import Input, Dense, Reshape, Flatten, Dropout
from keras.layers import BatchNormalization
from keras.layers import LeakyReLU
from keras.models import Sequential
from keras.optimizers import Adam

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['figure.figsize'] = (10, 5)
plt.style.use('ggplot')

Leemos datos MNIST

In [ ]:
# Datos
(X_train, _), (_, _) = mnist.load_data()
X_train = X_train.astype(np.float32) / 255.0 * 2.0 - 1.0
X_train = np.expand_dims(X_train, axis=3)

# Modelo

### Diseñamos modelo

Dimensiones y optimizador

In [ ]:
in_shape = X_train.shape
in_shape = in_shape[1:]
optimizador_gen = Adam(learning_rate=2e-4, beta_1=0.5, weight_decay=8e-8)
optimizador_dis = Adam(learning_rate=2e-4, beta_1=0.5, weight_decay=8e-8)

Modelo del Generador

In [ ]:
model_gen = Sequential(name='generator')
model_gen.add(Input((100,)))
model_gen.add(Dense(256))
model_gen.add(LeakyReLU(negative_slope=0.2))
model_gen.add(BatchNormalization(momentum=0.8))
model_gen.add(Dense(512))
model_gen.add(LeakyReLU(negative_slope=0.2))
model_gen.add(BatchNormalization(momentum=0.8))
model_gen.add(Dense(1024))
model_gen.add(LeakyReLU(negative_slope=0.2))
model_gen.add(BatchNormalization(momentum=0.8))
model_gen.add(Dense(int(np.prod(in_shape)), activation='tanh'))
model_gen.add(Reshape(in_shape))
model_gen.summary()

Modelo del discriminador

In [ ]:
model_disc = Sequential(name='discriminator')
model_disc.add(Input(in_shape))
model_disc.add(Flatten())
model_disc.add(Dense(128))
model_disc.add(LeakyReLU(negative_slope=0.2))
model_disc.add(Dense(64))
model_disc.add(LeakyReLU(negative_slope=0.2))
model_disc.add(Dense(1, activation='sigmoid'))
model_disc.summary()
model_disc.compile(loss='binary_crossentropy', optimizer=optimizador_dis, metrics=['accuracy'])

Modelo combinado

In [ ]:
# Important: freeze disc parameters before compiling the combined model
model_disc.trainable = False

model_gan = Sequential(name='gan')
model_gan.add(model_gen)
model_gan.add(model_disc)
model_gan.compile(loss='binary_crossentropy', optimizer=optimizador_gen)
model_gan.summary()

# Restore trainable state for discriminator
model_disc.trainable = True

# Entrenamiento

In [ ]:
DD_loss = []
GG_loss = []

In [ ]:
# Para acelerar los experimentos seleccionamos un subset de X_train
# - Con 10000 va rápido pero con solo 50 épocas no es suficiente
# - Con 30000 las curvas de entrenamiento empiezan a tener sentido
# - Con todo (60000) va muy lento, pero es lo que habría que hacer
n_train = 20000  # len(X_train)  # <= para el conjunto completo

# Parámetros del entrenamiento
epochs = 100
batch_size = 128
dataset = tf.data.Dataset.from_tensor_slices(X_train).\
    shuffle(len(X_train), reshuffle_each_iteration=True).batch(batch_size)

# Bucle entrenamiento
pb = tqdm(range(epochs), desc="Training")
for epoch in pb:
    g_loss = 0
    d_loss = 0
    d_accr = 0
    for x_real in dataset.take(n_train // batch_size):  # take retrieves # of batches
        bs = x_real.shape[0]  # current batch size

        ## Entrenamos discriminador
        # Imágenes sintéticas
        gen_noise = tf.random.normal((bs, 100))
        x_fake = model_gen(gen_noise)
        # Combinamos imágenes reales y sintéticas
        xx = tf.concat((x_real, x_fake), axis=0)
        yy = tf.concat((tf.ones((bs, 1)), tf.zeros((bs, 1))), axis=0)
        # Entrenamos discriminador
        model_disc.trainable = True
        d_metrics = model_disc.train_on_batch(xx, yy)
        d_loss += d_metrics[0]
        d_accr += d_metrics[1]

        ## Entrenamos generador
        model_disc.trainable = False
        noise = tf.random.normal((2*bs, 100))
        y_target = tf.ones((2*bs, 1))
        g_loss += model_gan.train_on_batch(noise, y_target)

    d_loss /= (n_train // batch_size)  # len(dataset)  # retorna el numero de lotes
    d_accr /= (n_train // batch_size)  # len(dataset)
    g_loss /= (n_train // batch_size)  # len(dataset)

    ## Evolución entrenamiento
    # if (epoch+1) % 10 == 0:
    #     tqdm.write(  # print(
    #         'Epoch: %5d, gen. loss: %.6f, disc. loss: %.6f, disc. acc: %.4f' %
    #         (epoch+1, g_loss, d_loss, d_accr))

    pb.set_description(
        f"Training gen. loss {g_loss:.6f}, disc. loss {d_loss:.6f} disc. acc: {d_accr:.4f}")

    DD_loss.append(d_loss)
    GG_loss.append(g_loss)

Curvas de aprendizaje

In [ ]:
plt.plot(DD_loss, label='Discriminator')
plt.plot(GG_loss, label='Generator')
plt.legend()
plt.show()

# Generar datos

In [ ]:
# Generamos imágenes sintéticas
gen_noise = np.random.normal(0, 1, (10, 100))
synthetic_images = model_gen(gen_noise)
# disc_prediction = np.round(model_disc(synthetic_images), 4)
disc_prediction = model_disc(synthetic_images).numpy() > 0.5

# Mostramos
half_len = len(synthetic_images) // 2
for i in range(half_len):
    plt.subplot(2, half_len, i+1)
    plt.imshow(synthetic_images[i], cmap='gray')
    plt.title(str(disc_prediction[i]))
    plt.axis('off')
    plt.subplot(2, half_len, half_len+i+1)
    plt.imshow(synthetic_images[2*i], cmap='gray')
    plt.title(str(disc_prediction[2*i]))
    plt.axis('off')
plt.subplots_adjust(hspace=-0.2)
plt.show()

## Imágenes reales para comparar

In [ ]:
half_len = 5
for i in range(half_len):
    plt.subplot(2, half_len, i+1)
    plt.imshow(X_train[i], cmap='gray')
    plt.axis('off')
    plt.subplot(2, half_len, half_len+i+1)
    plt.imshow(X_train[2*i], cmap='gray')
    plt.axis('off')
plt.subplots_adjust(hspace=-0.3)
plt.show()

# Ejercicios

### Ejercicio 1:
Modifica el código para que use el dataset CIFAR 10 en lugar del MNIST.

### Ejercicio 2:
Modifica el código para incluir la generación de las etiquetas.

### Ejercicio 3:
Modifica el código para que use capas convolucionales en lugar de densas (en la medida de lo posible).

### Ejercicio EXTRA:
Modifica el código para balancear cuanto aprende el generador y cuanto el discriminador.

Por ejemplo haz que el número de datos en el batch de entrenamiento dependa del coste (loss) en el paso de entrenamiento anterior de cada modelo (generador o discriminador). Es decir, si el discriminador aprende mucho (su loss es bajo) en el siguiente paso de entrenamiento le daré menos datos (para que aprenda menos).  

### Ejercicio EXTRA:
Crea un modelo que incluya todas las modificaciones de los cuatro ejercicios anteriores.